# XGLASS Training Demo

This notebook provides a minimal training demo for XGLASS. It loads the locally prepared projection dataset and gaze priors, instantiates the XGLASS slice relevance model, and runs the training loop to verify the pipeline. This notebook serves as a lightweight reproducibility and pipeline sanity check. For full-scale experiments and paper results, please use the CLI training script located at src/xglass/train.py.

**Data access & licensing note.** XGLASS relies on several third-party datasets, including resources that require controlled access (e.g., PhysioNet/REFLACX). This notebook does **not** download any datasets, does not request credentials, and does not redistribute any raw or derived data. It assumes you have already obtained all required datasets through their official channels, agreed to the corresponding licenses/DUAs, and prepared the files locally by following `../data/README.md`.

Which includes:

* A **projection dataset** folder (created by `data/create_projection_dataset.py` / your projection script) that contains:

  * `manifest.csv`
  * `data/` with split subfolders (`train/`, `val/`, `test/`) and per-sample folders containing at least:

    * `aip_wide.png` (and optionally `aip_lung.png`, `aip_soft.png`, `bl_cxr.png`)
    * `gt_data.json`
    * `bbox_256.json`

* A **splits file** (`balanced_splits.json`) listing the CT identifiers (`ct_name`) used for `train/`, `val/`, `test`.

* A **gaze importance maps** directory containing the disease-specific `.npz` files expected by the loader (e.g. `gaze_raw_atelectasis.npz`, …).

Here is the expected local folder structure:

```text
data/
└── processed/
    ├── projection_dataset/
    │   ├── manifest.csv
    │   └── data/
    │       ├── train/
    │       │   └── <category_slug>/
    │       │       └── <ct_stem>/
    │       │           ├── aip_wide.png
    │       │           ├── gt_data.json
    │       │           └── bbox_256.json
    │       ├── val/ ...
    │       └── test/ ...
    ├── gaze_importance_maps/
    │   ├── gaze_raw_atelectasis.npz
    │   ├── gaze_raw_consolidation.npz
    │   ├── ...
    └── balanced_splits.json # Here define your desired dataset split in train val and test.

```

## Setup

### Step 1 — Dependencies

Create and activate a fresh environment, then install all required packages:

In [11]:
%pip install -r ../requirements.txt

  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl.metadata (60 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl (12.9 MB)

  Attempting uninstall: numpy

    Found existing installation: numpy 2.3.5

    Uninstalling numpy-2.3.5:

      Successfully uninstalled numpy-2.3.5

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   --

  You can safely remove it manually.
  You can safely remove it manually.


### Step 2 — Import XGLASS from `src/`

In [ ]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
while not (p / "src" / "xglass").is_dir():
    if p.parent == p:
        raise RuntimeError("Run the notebook from inside the XGLASS repository.")
    p = p.parent
REPO_ROOT = p
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from xglass.dataset import SliceDataModule
from xglass.model import SlicePredictor

DATA_ROOT = REPO_ROOT / "data"
PROJ_ROOT = DATA_ROOT / "processed" / "projection_dataset"
HEATMAP_DIR = DATA_ROOT / "processed" / "gaze_importance_maps"
SPLITS_JSON = REPO_ROOT / "splits" / "paper_splits.json"

for path in (PROJ_ROOT / "manifest.csv", HEATMAP_DIR, SPLITS_JSON):
    if not path.exists():
        raise FileNotFoundError(f"Prepare the local data first (data/README.md): {path}")
    print(path)


xglass imported from: C:\Users\gl-mi\Documents\GitHub\XGLASS\src\xglass\__init__.py


c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the dataset and inspect one batch

In [ ]:
from xglass.dataset import SliceDataModule
import torch

dm = SliceDataModule(
    data_root=str(PROJ_ROOT),
    heatmap_dir=str(HEATMAP_DIR),
    splits_json=str(SPLITS_JSON),
    projection_type="aip_wide",
    blend_alpha=0.75,
    target_mode="area",
    batch_size=16,
    num_workers=0,   # increase if you want
)

dm.setup(stage="fit")
if len(dm.train_dataset) < 16 or len(dm.val_dataset) == 0:
    raise RuntimeError("Training needs the prepared paper train/validation data; the single-case inference example is insufficient.")
loader = dm.train_dataloader()
batch = next(iter(loader))

print("Batch elements:", len(batch))
for i, b in enumerate(batch):
    if torch.is_tensor(b):
        print(f"  batch[{i}] tensor shape={tuple(b.shape)} dtype={b.dtype}")
    else:
        print(f"  batch[{i}] type={type(b)}")


Pre-loading heatmaps from C:\Users\gl-mi\Documents\GitHub\XGLASS\data\processed\gaze_importance_maps...
Pre-loading heatmaps from C:\Users\gl-mi\Documents\GitHub\XGLASS\data\processed\gaze_importance_maps...
Pre-loading heatmaps from C:\Users\gl-mi\Documents\GitHub\XGLASS\data\processed\gaze_importance_maps...
Dataset setup complete:
  Train samples: 1
  Val samples:   1
  Test samples:  1
Batch elements: 3
  batch[0] tensor shape=(1, 3, 256, 256) dtype=torch.float32
  batch[1] tensor shape=(1, 240) dtype=torch.float32
  batch[2] tensor shape=(1, 240) dtype=torch.int64


c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


## Initialize the model and run a forward pass

In [35]:
import torch
from xglass.model import SlicePredictor

model = SlicePredictor(
    model_name="vit_base_patch16_224.augreg_in21k",
    num_slices=240,          # must match GT length
    learning_rate=1e-4,
    backbone_learning_rate=1e-5,
    backbone_frozen=False,
    data_image_size=256,
    cache_dir=None,
)

x = batch[0]  # [B,3,256,256]
with torch.no_grad():
    y = model(x)

print("Input:", tuple(x.shape))
print("Output:", tuple(y.shape))     # expect (1, 240)
print("Top-5 slices:", torch.topk(y[0], k=5).indices.cpu().tolist())


[Model Log] Attempting to create model: vit_base_patch16_224.augreg_in21k
[Model Log] Using cache directory: None
[Model Log] Successfully created backbone.
[Model Log] Backbone is FINETUNED. Training in end-to-end mode.
[Model Log] Creating resize transform for (256, 256) -> (224, 224)
Input: (1, 3, 256, 256)
Output: (1, 240)
Top-5 slices: [4, 47, 17, 22, 157]


c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\torchmetrics\utilities\prints.py:62: FutureWarning: Importing `RetrievalHitRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `RetrievalHitRate` from `torchmetrics.retrieval` instead.
  _future_warning(
c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


## Train the model

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

ckpt_dir = REPO_ROOT / "checkpoints" / "notebook_demo"
ckpt_dir.mkdir(parents=True, exist_ok=True)

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    log_every_n_steps=1,
    enable_checkpointing=True,
    callbacks=[
        ModelCheckpoint(dirpath=str(ckpt_dir), monitor="val_loss", mode="min", save_top_k=1),
        EarlyStopping(monitor="val_loss", mode="min", patience=3),
    ],
)

trainer.fit(model, datamodule=dm)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Pre-loading heatmaps from C:\Users\gl-mi\Documents\GitHub\XGLASS\data\processed\gaze_importance_maps...
Pre-loading heatmaps from C:\Users\gl-mi\Documents\GitHub\XGLASS\data\processed\gaze_importance_maps...
Pre-loading heatmaps from C:\Users\gl-mi\Documents\GitHub\XGLASS\data\processed\gaze_importance_maps...
Dataset setup complete:
  Train samples: 1
  Val samples:   1
  Test samples:  1
[Model Log] Configuring optimizer with differential LRs: Head=0.0001, Backbone=1e-05


c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

   | Name             | Type              | Params | Mode  | FLOPs
------------------------------------------------------------------------
0  | backbone         | VisionTransformer | 85.8 M | train | 0    
1  | resize_transform | Resize            | 0      | train | 0    
2  | head             | Sequential        | 516 K  | train | 0    
3  | loss_fn          | MSELoss           | 0      | train | 0    
4  | val_auc          | BinaryAUROC       |

c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\gl-mi\Documents\GitHub\XGLASS\.conda\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s, v_num=0, train_loss_step=0.171, val_loss=0.133, val_spearman_epoch=0.231, train_loss_epoch=0.171]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 1/1 [00:13<00:00,  0.07it/s, v_num=0, train_loss_step=0.171, val_loss=0.133, val_spearman_epoch=0.231, train_loss_epoch=0.171]
